# 06.5 — 未知时序概率模型：学习与推断

## 1. 问题定义

保持第 06 章的模型结构和数据生成方式不变，但不再向算法提供真实转移矩阵或观测模型参数。只给定有顺序的观测 $y_{0:T-1}$，已知状态数 $K=3$、初始分布均匀、每个状态的观测服从二维各向同性高斯分布。目标是学习 $\theta=(A,\mu,\sigma)$（各状态共享一个未知标准差），然后利用学到的模型推断状态。这里的学习和推断是两个不同的计算任务。

真实 x、A、均值及标准差仅用于仿真及评估。模型参数不必与真实值完全一致；隐状态的编号也可以任意置换。

## 2. 从贝叶斯滤波回到极大似然

第 06 章的更新分母为

$$e_t=p_\theta(y_t|y_{0:t-1})=\sum_k p_\theta(y_t|x_t=k)p_\theta(x_t=k|y_{0:t-1}).$$

$t=0$ 时历史为空。概率乘法公式给出

$$p_\theta(y_{0:T-1})=\prod_{t=0}^{T-1}e_t,\qquad
\log p_\theta(y_{0:T-1})=\sum_{t=0}^{T-1}\log e_t.$$

因此我们不需要真实状态标签，也无需枚举全部 $K^T$ 条状态轨迹。只要给定候选参数，执行一次滤波，就能计算其观测似然。学习目标与前五章一致：

$$\theta^*=\arg\max_\theta\log p_\theta(y_{0:T-1}).$$

第一步先学习单个转移参数，以便看清目标函数；第二步才学习全部指定参数。直接对准确的边缘似然求梯度，不必引入 EM 或变分推断。

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
from utils import plot_likelihood
rng = np.random.default_rng(42)
K, T = 3, 300
true_mu = np.array([[-2.,0.],[2.,0.],[0.,3.]])
true_sigma = 1.1
true_A = np.full((K,K), .025)
np.fill_diagonal(true_A, .95)
x = np.empty(T, dtype=int)
x[0] = rng.integers(K)
for t in range(1,T):
    x[t] = rng.choice(K, p=true_A[x[t-1]])
y = true_mu[x] + true_sigma*rng.normal(size=(T,2))

# 与第 06 章相同的滤波，只依赖候选模型和观测。
def filter_sequence(conditional, A):
    posterior = np.ones(K)/K
    filtered, log_evidence = [], []
    for t in range(len(conditional)):
        predicted = np.zeros(K)
        if t == 0:
            predicted = posterior.copy()
        else:
            for j in range(K):
                for i in range(K):
                    predicted[j] += A[i,j]*posterior[i]
        joint = conditional[t]*predicted
        evidence = joint.sum()
        posterior = joint/evidence
        filtered.append(posterior.copy())
        log_evidence.append(np.log(evidence))
    return np.array(filtered), np.array(log_evidence)

## 3. 最小学习实验：只估计自转移概率

本实验特意固定观测模型为真实模型，且限制 $A_{ii}=a$、$A_{ij}=(1-a)/2$（$i\ne j$）。唯一未知参数为 $a$。这是第 06 章原有的网格搜索实验，作为完全未知参数学习之前的过渡。

In [ ]:
conditional = np.column_stack([
    multivariate_normal(true_mu[k], true_sigma**2*np.eye(2)).pdf(y)
    for k in range(K)
])
candidates = np.linspace(.34,.995,100)
likelihood = []
for a in candidates:
    candidate = np.full((K,K),(1-a)/2)
    np.fill_diagonal(candidate,a)
    _, log_evidence = filter_sequence(conditional,candidate)
    likelihood.append(sum(log_evidence))
print('MLE of self-transition probability (grid):', candidates[np.argmax(likelihood)])
plot_likelihood(candidates, likelihood, .95, 'simulation truth')

## 4. 学习转移矩阵和观测模型

撤销上面的真实观测模型已知假设。参数由可训练变量表示：

- 转移矩阵每行经过 softmax，使行和为 1 且元素为正；
- 均值直接优化；
- 共享标准差由指数函数保证为正。

初始参数仅使用观测 y 和随机数：从观测中随机选择三个均值，转移初始值为均匀概率，标准差初始值为观测整体标准差。没有使用真实状态或真实生成参数。

为了避免极小概率的浮点下溢，下面使用对数域。设 $\ell_i=\log p(x_{t-1}=i|y_{0:t-1})$，则

$$\log p(x_t=j|y_{0:t-1})=\operatorname{logsumexp}_i(\ell_i+\log A_{ij}).$$

logsumexp 表示先取指数、求和、再取对数，只是边缘化的稳定计算方式。代码的 previous 维度是 i，current 维度是 j。预测后与观测对数密度相加，得到 log_joint；对当前状态求 logsumexp 得到 log_evidence；减去它就是归一化后的 log_posterior。

整个计算图保留梯度，由 PyTorch 对序列负对数似然求导。学习时每次参数变化都重新执行滤波，推断时固定最终参数。

In [ ]:
import torch
torch.manual_seed(42)
torch.set_num_threads(1)
obs = torch.tensor(y, dtype=torch.float64)
indices = torch.randperm(T)[:K]
mu = torch.nn.Parameter(obs[indices].clone())
transition_logits = torch.nn.Parameter(torch.zeros(K,K,dtype=obs.dtype))
log_sigma = torch.nn.Parameter(obs.std().log().detach())

def filter_log(log_conditional, log_A):
    log_posterior = torch.full((K,), -np.log(K), dtype=obs.dtype)
    filtered, log_evidences = [], []
    for t in range(len(log_conditional)):
        if t == 0:
            log_predicted = log_posterior
        else:
            # [i, j]：上一状态 i 的后验 × 从 i 到 j 的转移概率。
            log_joint_transition = log_posterior[:,None] + log_A
            log_predicted = torch.logsumexp(log_joint_transition, dim=0)
        log_joint = log_conditional[t] + log_predicted
        log_evidence = torch.logsumexp(log_joint, dim=0)
        log_posterior = log_joint - log_evidence
        filtered.append(log_posterior.exp())
        log_evidences.append(log_evidence)
    return torch.stack(filtered), torch.stack(log_evidences)

def model_filter():
    log_A = torch.log_softmax(transition_logits, dim=1)
    sigma = log_sigma.exp()
    # [T,K,2] -> [T,K]：对观测的两个独立维度累加 log density。
    models = torch.distributions.Normal(mu, sigma)
    log_conditional = models.log_prob(obs[:,None,:]).sum(dim=-1)
    return filter_log(log_conditional,log_A)

# 检查对数域实现与第 06 章的概率域实现一致。
with torch.no_grad():
    ref, ref_ll = filter_sequence(conditional,true_A)
    test, test_ll = filter_log(torch.tensor(conditional).log(),torch.tensor(true_A).log())
    assert np.allclose(ref,test.numpy())
    assert np.allclose(ref_ll,test_ll.numpy())

optimizer = torch.optim.Adam([mu, transition_logits, log_sigma],lr=.03)
losses = []
for step in range(250):
    optimizer.zero_grad()
    _, log_evidence = model_filter()
    loss = -log_evidence.mean()  # 除以 T 不改变 MLE 的最优点
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(step, loss.item())
plt.plot(losses)
plt.xlabel('optimization step')
plt.ylabel('negative log likelihood per observation')
plt.show()

## 5. 固定参数，进行状态推断与评估

训练后再次运行滤波，得到每个时刻的状态后验。真实状态编号与学习后的编号可能不同，因此仅在评估阶段对三个标签做全排列匹配；不将真实状态传入优化过程。

有限数据和非凸优化意味着不能保证恢复真实参数。此处只展示一次初始化的结果，不宣称全局最优。

In [ ]:
from itertools import permutations
with torch.no_grad():
    filtered, log_evidence = model_filter()
    learned_A = transition_logits.softmax(dim=1).numpy()
    learned_mu = mu.numpy().copy()
    learned_sigma = log_sigma.exp().item()
    assert np.allclose(filtered.sum(dim=1).numpy(),1)
    assert np.allclose(learned_A.sum(axis=1),1)
    print('final NLL:', -log_evidence.mean().item())

labels = filtered.argmax(dim=1).numpy()
# mapping[k] = 学到的状态 k 对应的真实状态编号。
mapping = max(permutations(range(K)),key=lambda p: (np.array(p)[labels]==x).mean())
aligned = np.array(mapping)[labels]
order = np.argsort(mapping)
print('aligned accuracy:', (aligned==x).mean())
print('learned A (aligned):\n', learned_A[np.ix_(order,order)])
print('true A:\n',true_A)
print('learned mu (aligned):\n',learned_mu[order])
print('true mu:\n',true_mu)
print('learned sigma:',learned_sigma,'true sigma:',true_sigma)
plt.figure(figsize=(10,3))
plt.plot(x[:100],label='truth')
plt.plot(aligned[:100],'--',label='inferred')
plt.legend()
plt.show()

## 6. 与第 05、07 章和 PlaNet 的联系

第 05 章的连续非线性隐变量模型难以直接计算边缘似然，因此学习近似后验并优化 ELBO。这里的离散状态数有限，滤波可以准确完成边缘化，因而可以直接 MLE，不需要 encoder。

第 07 章将离散状态改为连续线性高斯状态，利用卡尔曼滤波解析地完成相同的预测、更新及似然计算。对于 PlaNet 一类连续非线性时序隐变量模型，准确滤波和序列似然通常不可直接计算，因此需要将时序状态推断与变分学习结合。第 06.5 章尚未引入动作、神经网络或变分后验。